In [ ]:
# Stage2 model search + best-model tuning 

import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.metrics import classification_report, precision_recall_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

import lightgbm as lgb


LABEL = "fraud"
IDCOL = "id"

TRAIN_PATH = "../../DATA/dataset/TRAIN_stage2"
TEST_PATH  = "../../DATA/dataset/TEST_stage2"


def _require_cols(df, cols):
    miss = [c for c in cols if c not in df.columns]
    if miss:
        raise KeyError(f"Missing required columns: {miss}")


def best_f1_threshold_pr(y_true, score):
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score).astype(float)

    prec, rec, thr = precision_recall_curve(y_true, score)
    if len(thr) == 0:
        return 0.5

    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
    idx = int(np.nanargmax(f1))
    return float(thr[idx])


def fit_predict_eval(name, model, X_tr, y_tr, X_te, y_te, threshold="best_f1"):
    t0 = time.perf_counter()
    model.fit(X_tr, y_tr)
    fit_sec = time.perf_counter() - t0

    t1 = time.perf_counter()
    if hasattr(model, "predict_proba"):
        score = model.predict_proba(X_te)[:, 1].astype(float)
    else:
        score = model.decision_function(X_te).astype(float)
        score = 1.0 / (1.0 + np.exp(-score))
    pred_sec = time.perf_counter() - t1

    if threshold == "best_f1":
        thr = best_f1_threshold_pr(y_te, score)
    elif isinstance(threshold, (float, int)):
        thr = float(threshold)
    else:
        thr = 0.5

    y_pred = (score >= thr).astype(int)

    print("=" * 80)
    print(f"[{name}] thr={thr:.6f} fit={fit_sec:.3f}s pred={pred_sec:.3f}s")
    print(classification_report(y_te, y_pred, digits=4))

    rep = classification_report(y_te, y_pred, digits=6, output_dict=True)
    p1 = float(rep["1"]["precision"])
    r1 = float(rep["1"]["recall"])
    f1 = float(rep["1"]["f1-score"])
    s1 = int(rep["1"]["support"])

    return {
        "model": name,
        "fit_sec": fit_sec,
        "pred_sec": pred_sec,
        "thr": thr,
        "precision_1": p1,
        "recall_1": r1,
        "f1_1": f1,
        "support_1": s1,
    }


def make_models(random_state=42):
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ])

    models = []

    models.append((
        "logit",
        Pipeline([
            ("prep", num_pipe),
            ("clf", LogisticRegression(
                solver="saga",
                penalty="l2",
                C=1.0,
                class_weight="balanced",
                max_iter=2000,
                n_jobs=-1,
                random_state=random_state,
            )),
        ])
    ))

    models.append((
        "hgb",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", HistGradientBoostingClassifier(
                loss="log_loss",
                learning_rate=0.05,
                max_depth=None,
                max_iter=400,
                random_state=random_state,
            )),
        ])
    ))

    models.append((
        "rf",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=600,
                max_depth=None,
                min_samples_leaf=2,
                n_jobs=-1,
                random_state=random_state,
            )),
        ])
    ))

    models.append((
        "gnb",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", GaussianNB()),
        ])
    ))

    models.append((
        "lgb_small",
        lgb.LGBMClassifier(
            objective="binary",
            n_estimators=700,
            learning_rate=0.03,
            num_leaves=64,
            min_data_in_leaf=300,
            feature_fraction=0.9,
            subsample=0.9,
            subsample_freq=1,
            reg_lambda=1.0,
            class_weight="balanced",
            n_jobs=-1,
            random_state=random_state,
        )
    ))

    return models


train = pd.read_parquet(TRAIN_PATH)
test  = pd.read_parquet(TEST_PATH)

_require_cols(train, [IDCOL, LABEL])
_require_cols(test,  [IDCOL, LABEL])

feat_cols = [c for c in train.columns if c not in [IDCOL, LABEL]]
_require_cols(test, feat_cols)

X_tr = train[feat_cols]
y_tr = train[LABEL].astype(int).values

X_te = test[feat_cols]
y_te = test[LABEL].astype(int).values

results = []
models = make_models()

for name, model in tqdm(models, total=len(models)):
    res = fit_predict_eval(name, model, X_tr, y_tr, X_te, y_te, threshold="best_f1")
    results.append(res)

df_res = (
    pd.DataFrame(results)
      .sort_values(["f1_1", "recall_1", "precision_1", "pred_sec"], ascending=[False, False, False, True])
      .reset_index(drop=True)
)

df_res

  0%|          | 0/5 [00:00<?, ?it/s]

[logit] thr=0.000179 fit=10.429s pred=0.008s
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000       122
           1     0.8811    1.0000    0.9368       904

    accuracy                         0.8811      1026
   macro avg     0.4405    0.5000    0.4684      1026
weighted avg     0.7763    0.8811    0.8254      1026



/home/nakyung/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/n

[hgb] thr=0.312644 fit=2.957s pred=0.004s
              precision    recall  f1-score   support

           0     0.7091    0.9590    0.8153       122
           1     0.9942    0.9469    0.9700       904

    accuracy                         0.9483      1026
   macro avg     0.8516    0.9530    0.8927      1026
weighted avg     0.9603    0.9483    0.9516      1026

[rf] thr=0.312290 fit=35.910s pred=0.259s
              precision    recall  f1-score   support

           0     0.7383    0.9016    0.8118       122
           1     0.9863    0.9569    0.9714       904

    accuracy                         0.9503      1026
   macro avg     0.8623    0.9292    0.8916      1026
weighted avg     0.9568    0.9503    0.9524      1026

[gnb] thr=0.000000 fit=1.112s pred=0.002s
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000       122
           1     0.8811    1.0000    0.9368       904

    accuracy                         0.8811      1026
  

/home/nakyung/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/n

[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Info] Number of positive: 6568, number of negative: 601862
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012098 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1917
[LightGBM] [Info] Number of data points in the train set: 608430, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 

,model,fit_sec,pred_sec,thr,precision_1,recall_1,f1_1,support_1
0,rf,35.909722,0.258746,3.122902e-01,0.986317,0.956858,0.971364,904
1,hgb,2.957109,0.004041,3.126444e-01,0.994193,0.946903,0.969972,904
2,lgb_small,8.037617,0.003590,9.509298e-01,0.995338,0.944690,0.969353,904
3,gnb,1.112312,0.002153,1.095161e-34,0.881092,1.000000,0.936788,904
4,logit,10.429373,0.007748,1.785288e-04,0.881092,1.000000,0.936788,904


In [2]:
# best model 이름 확인
best_name = df_res.iloc[0]["model"]
best_name, df_res.iloc[0].to_dict()

('rf',
 {'model': 'rf',
  'fit_sec': 35.909721647854894,
  'pred_sec': 0.25874604494310915,
  'thr': 0.3122901843526844,
  'precision_1': 0.9863169897377423,
  'recall_1': 0.956858407079646,
  'f1_1': 0.9713644020213363,
  'support_1': 904})

In [ ]:
# Stage2: model compare + fit/pred time + classification_report + latency(p50/p95/p99)
# - saves fitted models
# - produces one summary df (metrics + timing + latency)
# - NO 3-way, binary only

import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.metrics import classification_report, precision_recall_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

import lightgbm as lgb


LABEL = "fraud"
IDCOL = "id"

TRAIN_PATH = "../../DATA/dataset/TRAIN_stage2"
TEST_PATH  = "../../DATA/dataset/TEST_stage2"


def _require_cols(df, cols):
    miss = [c for c in cols if c not in df.columns]
    if miss:
        raise KeyError(f"Missing required columns: {miss}")


def best_f1_threshold_pr(y_true, score):
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score).astype(float)

    prec, rec, thr = precision_recall_curve(y_true, score)
    if len(thr) == 0:
        return 0.5

    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
    idx = int(np.nanargmax(f1))
    return float(thr[idx])


def fit_predict_eval(name, model, X_tr, y_tr, X_te, y_te, threshold="best_f1"):
    t0 = time.perf_counter()
    model.fit(X_tr, y_tr)
    fit_sec = time.perf_counter() - t0

    t1 = time.perf_counter()
    if hasattr(model, "predict_proba"):
        score = model.predict_proba(X_te)[:, 1].astype(float)
    else:
        score = model.decision_function(X_te).astype(float)
        score = 1.0 / (1.0 + np.exp(-score))
    pred_sec = time.perf_counter() - t1

    if threshold == "best_f1":
        thr = best_f1_threshold_pr(y_te, score)
    elif isinstance(threshold, (float, int)):
        thr = float(threshold)
    else:
        thr = 0.5

    y_pred = (score >= thr).astype(int)

    print("=" * 80)
    print(f"[{name}] thr={thr:.6f} fit={fit_sec:.3f}s pred={pred_sec:.3f}s")
    print(classification_report(y_te, y_pred, digits=4, zero_division=0))

    rep = classification_report(y_te, y_pred, digits=6, output_dict=True, zero_division=0)

    p1 = float(rep["1"]["precision"])
    r1 = float(rep["1"]["recall"])
    f1v = float(rep["1"]["f1-score"])
    s1 = int(rep["1"]["support"])

    p0 = float(rep["0"]["precision"])
    r0 = float(rep["0"]["recall"])
    f0 = float(rep["0"]["f1-score"])
    s0 = int(rep["0"]["support"])

    acc = float(rep["accuracy"])

    return score, y_pred, {
        "model": name,
        "fit_sec": float(fit_sec),
        "pred_sec": float(pred_sec),
        "thr": float(thr),
        "accuracy": acc,
        "precision_1": p1,
        "recall_1": r1,
        "f1_1": f1v,
        "support_1": s1,
        "precision_0": p0,
        "recall_0": r0,
        "f1_0": f0,
        "support_0": s0,
    }


def latency_bench_single_np(model, X_np, n_warmup=50, n_iter=800, rng=42):
    rng = np.random.default_rng(rng)
    n = len(X_np)
    if n == 0:
        return {
            "lat_mean_ms": np.nan,
            "lat_p50_ms": np.nan,
            "lat_p95_ms": np.nan,
            "lat_p99_ms": np.nan,
            "lat_max_ms": np.nan,
        }

    idx = rng.integers(0, n, size=n_iter)

    # warmup
    for i in range(min(n_warmup, n_iter)):
        xi = X_np[int(idx[i])].reshape(1, -1)
        _ = model.predict_proba(xi)[:, 1]

    ts = np.empty(n_iter, dtype=np.float64)
    for i in range(n_iter):
        xi = X_np[int(idx[i])].reshape(1, -1)
        t0 = time.perf_counter()
        _ = model.predict_proba(xi)[:, 1]
        ts[i] = time.perf_counter() - t0

    return {
        "lat_mean_ms": float(ts.mean() * 1000),
        "lat_p50_ms": float(np.quantile(ts, 0.50) * 1000),
        "lat_p95_ms": float(np.quantile(ts, 0.95) * 1000),
        "lat_p99_ms": float(np.quantile(ts, 0.99) * 1000),
        "lat_max_ms": float(ts.max() * 1000),
    }


def make_models(random_state=42):
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
    ])

    models = []

    models.append((
        "logit",
        Pipeline([
            ("prep", num_pipe),
            ("clf", LogisticRegression(
                solver="saga",
                penalty="l2",
                C=1.0,
                class_weight="balanced",
                max_iter=3000,
                n_jobs=-1,
                random_state=random_state,
            )),
        ])
    ))

    models.append((
        "hgb",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", HistGradientBoostingClassifier(
                loss="log_loss",
                learning_rate=0.05,
                max_depth=None,
                max_iter=500,
                random_state=random_state,
            )),
        ])
    ))

    models.append((
        "rf",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=800,
                max_depth=None,
                min_samples_leaf=2,
                n_jobs=-1,
                random_state=random_state,
            )),
        ])
    ))

    models.append((
        "gnb",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", GaussianNB()),
        ])
    ))

    models.append((
        "lgb_small",
        lgb.LGBMClassifier(
            objective="binary",
            n_estimators=800,
            learning_rate=0.03,
            num_leaves=64,
            min_data_in_leaf=300,
            feature_fraction=0.9,
            subsample=0.9,
            subsample_freq=1,
            reg_lambda=1.0,
            class_weight="balanced",
            n_jobs=-1,
            random_state=random_state,
            force_row_wise=True,
        )
    ))

    return models

# Load

train = pd.read_parquet(TRAIN_PATH)
test  = pd.read_parquet(TEST_PATH)

_require_cols(train, [IDCOL, LABEL])
_require_cols(test,  [IDCOL, LABEL])

feat_cols = [c for c in train.columns if c not in [IDCOL, LABEL]]
_require_cols(test, feat_cols)

X_tr = train[feat_cols]
y_tr = train[LABEL].astype(int).values

X_te = test[feat_cols]
y_te = test[LABEL].astype(int).values

# Fit/Eval all

results = []
fitted = {}
scores = {}
preds = {}

models = make_models()

for name, model in tqdm(models, total=len(models)):
    score_te, y_pred, row = fit_predict_eval(name, model, X_tr, y_tr, X_te, y_te, threshold="best_f1")
    fitted[name] = model
    scores[name] = score_te
    preds[name] = y_pred
    results.append(row)


# Summary df 
df_res = pd.DataFrame(results)

for c in ["lat_mean_ms", "lat_p50_ms", "lat_p95_ms", "lat_p99_ms", "lat_max_ms"]:
    df_res[c] = np.nan

df_res = (
    df_res
      .sort_values(["f1_1", "recall_1", "precision_1", "pred_sec"], ascending=[False, False, False, True])
      .reset_index(drop=True)
)


# Latency bench 
X_te_np = X_te.values

for i, row in tqdm(df_res.iterrows(), total=len(df_res)):
    name = row["model"]
    model = fitted[name]
    try:
        lat = latency_bench_single_np(model, X_te_np, n_warmup=50, n_iter=800, rng=42)
    except Exception:
        lat = {
            "lat_mean_ms": np.nan,
            "lat_p50_ms": np.nan,
            "lat_p95_ms": np.nan,
            "lat_p99_ms": np.nan,
            "lat_max_ms": np.nan,
        }
    for k, v in lat.items():
        df_res.loc[i, k] = v


# final rank: use p95 latency as tie-breaker
df_res = (
    df_res
      .sort_values(["f1_1", "recall_1", "precision_1", "lat_p95_ms"], ascending=[False, False, False, True])
      .reset_index(drop=True)
)

df_res

  0%|          | 0/5 [00:00<?, ?it/s]

[logit] thr=0.000179 fit=10.681s pred=0.008s
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000       122
           1     0.8811    1.0000    0.9368       904

    accuracy                         0.8811      1026
   macro avg     0.4405    0.5000    0.4684      1026
weighted avg     0.7763    0.8811    0.8254      1026

[hgb] thr=0.312644 fit=2.603s pred=0.004s
              precision    recall  f1-score   support

           0     0.7091    0.9590    0.8153       122
           1     0.9942    0.9469    0.9700       904

    accuracy                         0.9483      1026
   macro avg     0.8516    0.9530    0.8927      1026
weighted avg     0.9603    0.9483    0.9516      1026

[rf] thr=0.335004 fit=45.103s pred=0.316s
              precision    recall  f1-score   support

           0     0.7368    0.9180    0.8175       122
           1     0.9886    0.9558    0.9719       904

    accuracy                         0.9513      1026

  0%|          | 0/5 [00:00<?, ?it/s]

/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/home

[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGB

...

[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGB

/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGB

/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
/home

,model,fit_sec,pred_sec,thr,accuracy,precision_1,recall_1,f1_1,support_1,precision_0,recall_0,f1_0,support_0,lat_mean_ms,lat_p50_ms,lat_p95_ms,lat_p99_ms,lat_max_ms
0,rf,45.102811,0.315958,3.350041e-01,0.951267,0.988558,0.955752,0.971879,904,0.736842,0.918033,0.817518,122,256.632904,259.396595,290.553589,330.966967,348.604851
1,hgb,2.603003,0.004030,3.126444e-01,0.948343,0.994193,0.946903,0.969972,904,0.709091,0.959016,0.815331,122,1.976159,1.952514,2.036340,2.462131,3.233239
2,lgb_small,6.599943,0.004548,9.452566e-01,0.947368,0.995338,0.944690,0.969353,904,0.702381,0.967213,0.813793,122,1.020293,0.955686,1.023906,2.798677,6.893910
3,gnb,1.112833,0.002183,1.095161e-34,0.881092,0.881092,1.000000,0.936788,904,0.000000,0.000000,0.000000,122,0.458667,0.456932,0.472645,0.484921,0.583602
4,logit,10.681195,0.008016,1.785288e-04,0.881092,0.881092,1.000000,0.936788,904,0.000000,0.000000,0.000000,122,0.595491,0.591539,0.616408,0.742120,0.819346


RF는 “성능 미세 우위”지만

latency 100배 느림

학습 시간 45초

모델 크기 큼

scale-out 시 비용 증가


---
**[최적 모델]**

***lgb_small***


F1 거의 RF와 동일

latency 1ms

p95 매우 안정적

메모리 효율 좋음

배포 친화적

---

In [ ]:
# Fast LGB tuning (random search + early stopping)

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, precision_recall_curve

import lightgbm as lgb

LABEL = "fraud"
IDCOL = "id"

TRAIN_PATH = "../../DATA/dataset/TRAIN_stage2"
TEST_PATH  = "../../DATA/dataset/TEST_stage2"

OUT_DIR = Path("../../DATA/stage2_outputs/lgb_tuning_fast")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_PATH = OUT_DIR / "lgb_tuning_fast_summary.csv"
BEST_MODEL_PATH = OUT_DIR / "best_lgb_model.txt"
BEST_JSON_PATH  = OUT_DIR / "best_lgb_meta.json"


def _require_cols(df, cols):
    miss = [c for c in cols if c not in df.columns]
    if miss:
        raise KeyError(f"Missing required columns: {miss}")


def best_f1_threshold_pr(y_true, score):
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score).astype(float)

    prec, rec, thr = precision_recall_curve(y_true, score)
    if len(thr) == 0:
        return 0.5

    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
    idx = int(np.nanargmax(f1))
    return float(thr[idx])


def latency_bench_predict_proba(model, X_np, n_warmup=50, n_iter=400, seed=42):
    rng = np.random.default_rng(seed)
    n = len(X_np)
    if n == 0:
        return {"lat_mean_ms": np.nan, "lat_p50_ms": np.nan, "lat_p95_ms": np.nan, "lat_p99_ms": np.nan, "lat_max_ms": np.nan}

    idx = rng.integers(0, n, size=n_iter)

    for i in range(min(n_warmup, n_iter)):
        xi = X_np[int(idx[i])].reshape(1, -1)
        _ = model.predict_proba(xi)[:, 1]

    ts = np.empty(n_iter, dtype=np.float64)
    for i in range(n_iter):
        xi = X_np[int(idx[i])].reshape(1, -1)
        t0 = time.perf_counter()
        _ = model.predict_proba(xi)[:, 1]
        ts[i] = time.perf_counter() - t0

    return {
        "lat_mean_ms": float(ts.mean() * 1000),
        "lat_p50_ms": float(np.quantile(ts, 0.50) * 1000),
        "lat_p95_ms": float(np.quantile(ts, 0.95) * 1000),
        "lat_p99_ms": float(np.quantile(ts, 0.99) * 1000),
        "lat_max_ms": float(ts.max() * 1000),
    }


def sample_params(rng):
    return {
        "learning_rate": float(rng.choice([0.01, 0.02, 0.03, 0.05])),
        "num_leaves": int(rng.choice([31, 63, 127, 255])),
        "min_data_in_leaf": int(rng.choice([80, 150, 300, 600])),
        "feature_fraction": float(rng.choice([0.7, 0.8, 0.9, 1.0])),
        "subsample": float(rng.choice([0.7, 0.8, 0.9, 1.0])),
        "subsample_freq": 1,
        "reg_lambda": float(rng.choice([0.0, 0.5, 1.0, 3.0, 10.0])),
        "reg_alpha": float(rng.choice([0.0, 0.1, 0.5, 1.0])),
        "max_depth": int(rng.choice([-1, 6, 10, 14])),
    }


def fit_eval_valid(params, X_tr, y_tr, X_va, y_va, n_estimators=4000, early_stopping_rounds=100):
    model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=int(n_estimators),
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
        force_row_wise=True,
        **params,
    )

    t0 = time.perf_counter()
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="binary_logloss",
        callbacks=[lgb.early_stopping(stopping_rounds=int(early_stopping_rounds), verbose=False)],
    )
    fit_sec = time.perf_counter() - t0

    t1 = time.perf_counter()
    score_va = model.predict_proba(X_va)[:, 1].astype(float)
    pred_sec = time.perf_counter() - t1

    thr = best_f1_threshold_pr(y_va, score_va)
    y_pred_va = (score_va >= thr).astype(int)

    rep = classification_report(y_va, y_pred_va, output_dict=True, zero_division=0)
    f1_1 = float(rep["1"]["f1-score"])
    r1 = float(rep["1"]["recall"])
    p1 = float(rep["1"]["precision"])
    acc = float(rep["accuracy"])

    best_iter = int(getattr(model, "best_iteration_", 0) or 0)

    row = {
        "fit_sec": float(fit_sec),
        "pred_sec": float(pred_sec),
        "thr_valid": float(thr),
        "valid_accuracy": acc,
        "valid_precision_1": p1,
        "valid_recall_1": r1,
        "valid_f1_1": f1_1,
        "best_iteration": best_iter,
        **params,
    }
    return model, score_va, y_pred_va, row


def eval_on_test(model, X_te, y_te, threshold="best_f1"):
    t1 = time.perf_counter()
    score = model.predict_proba(X_te)[:, 1].astype(float)
    pred_sec = time.perf_counter() - t1

    if threshold == "best_f1":
        thr = best_f1_threshold_pr(y_te, score)
    else:
        thr = float(threshold)

    y_pred = (score >= thr).astype(int)

    print("=" * 80)
    print(f"[BEST TEST] thr={thr:.6f} pred={pred_sec:.3f}s")
    print(classification_report(y_te, y_pred, digits=4, zero_division=0))

    rep = classification_report(y_te, y_pred, output_dict=True, zero_division=0)
    out = {
        "test_pred_sec": float(pred_sec),
        "thr_test": float(thr),
        "test_accuracy": float(rep["accuracy"]),
        "test_precision_1": float(rep["1"]["precision"]),
        "test_recall_1": float(rep["1"]["recall"]),
        "test_f1_1": float(rep["1"]["f1-score"]),
        "test_precision_0": float(rep["0"]["precision"]),
        "test_recall_0": float(rep["0"]["recall"]),
        "test_f1_0": float(rep["0"]["f1-score"]),
    }
    return score, y_pred, out


# ---------------- load ----------------
train = pd.read_parquet(TRAIN_PATH)
test  = pd.read_parquet(TEST_PATH)

_require_cols(train, [IDCOL, LABEL])
_require_cols(test,  [IDCOL, LABEL])

feat_cols = [c for c in train.columns if c not in [IDCOL, LABEL]]
_require_cols(test, feat_cols)

X_raw = train[feat_cols]
y = train[LABEL].astype(int).values

X_te_raw = test[feat_cols]
y_te = test[LABEL].astype(int).values

imp = SimpleImputer(strategy="median")
X_all = imp.fit_transform(X_raw)
X_te = imp.transform(X_te_raw)

X_tr, X_va, y_tr, y_va = train_test_split(
    X_all, y, test_size=0.15, random_state=42, stratify=y
)

# ---------------- random tuning ----------------
rng = np.random.default_rng(42)

N_TRIALS = 40
N_ESTIMATORS = 4000
EARLY_STOP = 100

rows = []
best_key = None
best_pack = None

for t in tqdm(range(N_TRIALS), total=N_TRIALS):
    params = sample_params(rng)

    model, score_va, y_pred_va, row = fit_eval_valid(
        params, X_tr, y_tr, X_va, y_va,
        n_estimators=N_ESTIMATORS,
        early_stopping_rounds=EARLY_STOP
    )

    rows.append(row)

    print("=" * 80)
    print(f"[trial={t+1}/{N_TRIALS}] valid_f1_1={row['valid_f1_1']:.6f} "
          f"r1={row['valid_recall_1']:.6f} p1={row['valid_precision_1']:.6f} "
          f"fit={row['fit_sec']:.3f}s best_iter={row['best_iteration']}")

    key = (row["valid_f1_1"], row["valid_recall_1"], row["valid_precision_1"], -row["fit_sec"])
    if (best_key is None) or (key > best_key):
        best_key = key
        best_pack = (model, params, row)

df_sum = pd.DataFrame(rows).sort_values(
    ["valid_f1_1", "valid_recall_1", "valid_precision_1", "fit_sec"],
    ascending=[False, False, False, True]
).reset_index(drop=True)

df_sum.to_csv(SUMMARY_PATH, index=False)
print("saved summary:", str(SUMMARY_PATH))

best_model, best_params, best_row = best_pack

print("=" * 80)
print("[BEST ON VALID]")
print(pd.Series(best_row))

# ---------------- final test report
_ = eval_on_test(best_model, X_te, y_te, threshold="best_f1")

# ---------------- latency bench 
lat = latency_bench_predict_proba(best_model, X_te, n_warmup=50, n_iter=400, seed=42)
print("=" * 80)
print("[BEST LATENCY ms]")
print(pd.Series(lat))

best_model.booster_.save_model(str(BEST_MODEL_PATH))
with open(BEST_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump({"best_params": best_params, "best_row_valid": best_row, "latency_ms": lat}, f, ensure_ascii=False, indent=2)

print("saved best model:", str(BEST_MODEL_PATH))
print("saved meta json:", str(BEST_JSON_PATH))

  0%|          | 0/40 [00:00<?, ?it/s]

[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Info] Number of positive: 5583, number of negative: 511582
[LightGBM] [Info] Total Bins 1918
[LightGBM] [Info] Number of data points in the train set: 517165, number of used features: 21
[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 

/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[trial=1/40] valid_f1_1=0.950495 r1=0.925888 p1=0.976445 fit=49.390s best_iter=4000
[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Warning] min_data_in_leaf is set=300, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=300
[LightGBM] [Warning] feature_fraction is set=1.0, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=1.0
[LightGBM] [Info] Number of positive: 5583, number of negative: 511582
[LightGBM] [Info] Total Bins 1918
[LightGBM] [Info] Number of data points in the train set: 517165, n

/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=600, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=600
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] min_data_in_leaf is set=600, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=600
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] min_data_in_leaf is set=600, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=600
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] min_data_in_leaf is set=600, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=600
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGB

/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=600, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=600
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] min_data_in_leaf is set=600, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=600
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] min_data_in_leaf is set=600, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=600
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] min_data_in_leaf is set=600, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=600
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGB

/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/nakyung/.local/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
LABEL = "fraud"
IDCOL = "id"

TEST_PATH = "../../DATA/dataset/TEST_stage2"
BEST_MODEL_PATH = "../../DATA/stage2_outputs/lgb_tuning_fast/best_lgb_model.txt"
BEST_JSON_PATH  = "../../DATA/stage2_outputs/lgb_tuning_fast/best_lgb_meta.json"


def best_f1_threshold_pr(y_true, score):
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score).astype(float)
    prec, rec, thr = precision_recall_curve(y_true, score)
    if len(thr) == 0:
        return 0.5
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
    idx = int(np.nanargmax(f1))
    return float(thr[idx])


# 1) load test
test = pd.read_parquet(TEST_PATH)
feat_cols = [c for c in test.columns if c not in [IDCOL, LABEL]]

X_te_raw = test[feat_cols]
y_te = test[LABEL].astype(int).values

# 2) impute 
imp = SimpleImputer(strategy="median")
X_te = imp.fit_transform(X_te_raw)

# 3) load model + meta
booster = lgb.Booster(model_file=BEST_MODEL_PATH)
with open(BEST_JSON_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

# 4) predict score
score_te = booster.predict(X_te).astype(float)

# 5) threshold 선택
thr = best_f1_threshold_pr(y_te, score_te)

y_pred = (score_te >= thr).astype(int)

print("=" * 80)
print(f"[BEST LGB REPORT] thr={thr:.6f}")
print(classification_report(y_te, y_pred, digits=4, zero_division=0))

[BEST LGB REPORT] thr=0.808158
              precision    recall  f1-score   support

           0     0.7303    0.9098    0.8102       122
           1     0.9874    0.9546    0.9708       904

    accuracy                         0.9493      1026
   macro avg     0.8588    0.9322    0.8905      1026
weighted avg     0.9568    0.9493    0.9517      1026

